In [12]:
BASE_PATH = r"CSCI 5980 8980 Project"

ORIGIN_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/"
OUTPUT_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/eval-1k/"

layers_full_path = os.path.join(OUTPUT_PATH, "layers_full.csv")

summary_full_path = os.path.join(OUTPUT_PATH, "summary_full.csv")

In [2]:
# drive allocation
import os
from google.colab import drive
from datasets import load_dataset, Dataset, load_from_disk

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd, json, os

In [14]:
summaryA = os.path.join(ORIGIN_PATH, "lite_results_summary_judged_trek.csv")
summaryB  = os.path.join(ORIGIN_PATH, "results_summary_judged_brandon.csv")

In [19]:
dfA = pd.read_csv(summaryA)
dfB = pd.read_csv(summaryB)

# combine dfA and dfB
df = pd.concat([dfA, dfB])

summary_full = os.path.join(OUTPUT_PATH, "summary_full.csv")

df.to_csv(summary_full, index=False)

In [9]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split

# 1. Load Data
layers_full = pd.read_csv(layers_full_path)
summary_full = pd.read_csv(summary_full_path)

# 2. Pivot & Merge
layer_pivot = (
    layers_full.pivot_table(index="question_id", columns="layer", values="f1")
    .sort_index(axis=1)
    .reset_index()
)
layer_pivot.columns = ["question_id"] + [f"f1_layer_{c}" for c in layer_pivot.columns[1:]]

df = summary_full.merge(layer_pivot, on="question_id")

# 3. Define Features and Stratification
layer_cols = [c for c in df.columns if c.startswith("f1_layer_")]
feature_cols = layer_cols + ["normalized_log_confidence"]

# Create a combined key for balanced 'type' AND 'judge_decision'
df['stratify_key'] = df['type'].astype(str) + "_" + df['judge_decision'].astype(str)

# 4. Stratified Splitting
# Split 1: 80% Train+Val, 20% Test
df_tv, df_test = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['stratify_key']
)

# Split 2: Of that 80%, 25% for Val (results in 60% Train, 20% Val total)
df_train, df_val = train_test_split(
    df_tv, test_size=0.25, random_state=42, stratify=df_tv['stratify_key']
)

# 5. Extract Arrays
def get_xy(target_df):
    X = target_df[feature_cols].values.astype(np.float32)
    y = target_df["judge_decision"].values.astype(int)
    return X, y

X_train, y_train = get_xy(df_train)
X_val, y_val = get_xy(df_val)
X_test, y_test = get_xy(df_test)

# 6. Save Bundle with Joblib
data_bundle = {
    'train': (X_train, y_train),
    'val': (X_val, y_val),
    'test': (X_test, y_test),
    'feature_names': feature_cols,
    'metadata': {
        'train_ids': df_train['question_id'].values,
        'val_ids': df_val['question_id'].values,
        'test_ids': df_test['question_id'].values
    }
}

bundle_path = os.path.join(OUTPUT_PATH, "data_bundle.joblib")
joblib.dump(data_bundle, bundle_path)

print(f"Data bundle saved successfully to: {bundle_path}")
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Data bundle saved successfully to: /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/MICE_Output/finalizer/data_bundle.joblib
Train size: 600, Val size: 200, Test size: 200
